# Lab 7: Generational Drift Dashboards & UMAP
**Course:** CIS 531/731: Programming Techniques for Data Science and Analytics
**Date:** Friday, October 23, 2026
**Module:** 7 (Explore: Data Visualization & Dashboards)

This lab bridges the theoretical mathematics of UMAP with interactive data storytelling. You will extract chronologically sequenced embeddings, establish a global topological coordinate space, quantify semantic drift, and inject the pipeline into an interactive dashboard.

## Part 1: Environment Setup & Data Extraction
First, we import our required libraries. Ensure you have installed `umap-learn`, `hdbscan`, `plotly`, and `streamlit` in your DevContainer.

Your data must be sorted chronologically. In a real pipeline, you would query your Vector Database (VDB) for embeddings matching specific temporal metadata.

In [ ]:
import pandas as pd
import numpy as np
import umap
import hdbscan
import plotly.express as px
from scipy.spatial.distance import cosine

# Placeholder for VDB Extraction (The 'Obtain/Scrub' Handoff)
# df = pd.read_parquet("silver_embeddings_with_timestamps.parquet")

# For this lab, we mock a DataFrame containing 768-D embeddings, text, and eras
np.random.seed(42)
mock_embeddings = np.random.rand(1000, 768)
eras = np.random.choice(['1990s', '2000s', '2010s', '2020s'], 1000)

df = pd.DataFrame({'era': eras, 'text_snippet': ['sample text'] * 1000})
# In practice, keep your high-dimensional vectors in a separate numpy array or tensor
embeddings = mock_embeddings

## Part 2: Global UMAP Projection
**CRITICAL ARCHITECTURAL RULE:** You must fit your UMAP model on the *entire* dataset globally before slicing by time. If you fit UMAP on each chronological slice independently, the coordinate space will warp, rendering drift metrics mathematically meaningless.

In [ ]:
# 1. Initialize the UMAP Reducer
# Tune n_neighbors for global vs. local structure and min_dist for cluster density
reducer = umap.UMAP(
    n_neighbors=15, 
    min_dist=0.1, 
    n_components=2, 
    random_state=42
)

# 2. Fit and transform the GLOBAL dataset
print("Fitting global UMAP topology...")
global_2d_projections = reducer.fit_transform(embeddings)

# 3. Bind the 2D coordinates back to the metadata for the dashboard
df['umap_x'] = global_2d_projections[:, 0]
df['umap_y'] = global_2d_projections[:, 1]

## Part 3: Quantifying Generational Drift
Visualizing the shift is excellent for EDA, but we must back it up with math. We will isolate the embeddings for a specific semantic cluster in the 1990s and the 2020s, calculate their centroids in the *high-dimensional space* (768-D), and measure the cosine distance.

In [ ]:
def calculate_centroid_drift(embeddings_matrix, metadata_df, era_a, era_b):
    """
    Calculates the cosine distance between the centroids of two temporal slices.
    """
    # Extract indices for each era
    idx_a = metadata_df.index[metadata_df['era'] == era_a].tolist()
    idx_b = metadata_df.index[metadata_df['era'] == era_b].tolist()
    
    # Calculate high-dimensional centroids (mean across the 768 dimensions)
    centroid_a = np.mean(embeddings_matrix[idx_a], axis=0)
    centroid_b = np.mean(embeddings_matrix[idx_b], axis=0)
    
    # Calculate Cosine Distance (0 = identical, 1 = orthogonal, 2 = opposite)
    drift_distance = cosine(centroid_a, centroid_b)
    return drift_distance

drift_score = calculate_centroid_drift(embeddings, df, '1990s', '2020s')
print(f"Semantic Drift (Cosine Distance) from 1990s to 2020s: {drift_score:.4f}")

## Part 4: Interactive Dashboard Architecture (Streamlit)
Below is the skeleton for your `app.py` script. You cannot run Streamlit directly inside this Jupyter Cell. You must export this code to a `.py` file and run `streamlit run app.py` from your terminal.

The callback architecture filters the dataframe based on the slider and re-renders the Plotly scatter plot dynamically.

In [ ]:
"""
# Save this cell as app.py and run via CLI
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("Generational Drift Explorer")
st.markdown("**Note:** Ensure your data is fully projected via UMAP before loading it here.")

# 1. Load the pre-processed dataframe (containing umap_x, umap_y, era, and text)
# @st.cache_data
# def load_data():
#     return pd.read_parquet('dashboard_ready_data.parquet')
# df = load_data()

# 2. Build the UI Filter (Time Slider or Selectbox)
# selected_era = st.select_slider('Select Era', options=['1990s', '2000s', '2010s', '2020s'])

# 3. Execute the Callback / Filter
# filtered_df = df[df['era'] == selected_era]

# 4. Render the Plotly Chart
# fig = px.scatter(
#     filtered_df, 
#     x='umap_x', 
#     y='umap_y', 
#     hover_data=['text_snippet'], # Tooltip binding
#     title=f"Semantic Space in the {selected_era}"
# )
# st.plotly_chart(fig, use_container_width=True)
"""

## Part 5: MP6 Checkpoint & MP7 Deliverables

**🚨 Action Items:**
1. **MP6 Checkpoint:** Verify that your Model Selection baseline code is committed to your repository. 
2. **MP7 Initialization:** MP7 is explicitly focused on Exploratory Dashboards and Data Storytelling. You must deploy an interactive dashboard (using Streamlit or Dash) that visualizes a core dimension of your term project dataset.
3. **Artifact Gate:** Your MP7 submission must include the GitOps PR and a link to the deployed/runnable dashboard instance.